# version14 板端 PYNQ 检查与运行
本 Notebook 与常驻入口使用同一套 Session/PynqOwner/C 核心。电脑网页及串口协议保持不变。前两步只读文件；运行单元会操作真实板卡，必须已有本次授权及当前 boot 的受保护准入回执。没有回执不会自动重置、加载或分配。

In [ ]:
from pathlib import Path
import sys, json
for root in [Path.cwd()] + list(Path.cwd().parents):
    if (root / 'pynq_vision/beam_launcher.py').is_file():
        sys.path.insert(0, str(root))
        break
else:
    raise RuntimeError('请在完整交付目录中打开 Notebook')
from pynq_vision.beam_launcher import Session
BUNDLE = root / 'output/build/game_app/pynq_beam_hardware_3'
UART_DEVICE = None  # 填写已确认且不再作 Linux 控制台的板上 UART
I2C_DEVICE = None   # 填写已确认映射到 PS I2C0 的 /dev/i2c-*
session = Session(BUNDLE/'hardware.json', BUNDLE/'system_wrapper.bit',
                  BUNDLE/'libbeam_pynq.so', UART_DEVICE, I2C_DEVICE)
session.inspect()

## 查看原工程参数
以下参数从冻结 XSA 和当前 C 相机表派生，不在 Notebook 中复制或调整游戏阈值。

In [ ]:
hardware = json.loads((BUNDLE/'hardware.json').read_text())
{'frame': hardware['frame'], 'fclk_mhz': hardware['fclk_mhz'],
 'csi_version': hex(hardware['csi_version']),
 'sensor_table_entries': [len(hardware[k]) for k in ('sensor_init', 'sensor_mode', 'sensor_expected')]} 

## 实际采集、HDMI 和串口运行
先填写设备路径、按仓库流程完成当前 boot/独占/停机准入，再执行此单元。`max_frames=10` 是有界硬件实验；正常游戏改为 `max_frames=0` 持续运行。中断单元会进入同一安全停机流程，未确认 DMA HALTED 时进程会保留资源，不要强制杀内核。旧 BOOT 不会被此 Notebook 写入。

In [ ]:
# 实际板卡操作：仅在本轮授权和准入均完成后执行
assert UART_DEVICE and I2C_DEVICE, '先填写并确认 UART/I2C 设备路径'
session = Session(BUNDLE/'hardware.json', BUNDLE/'system_wrapper.bit',
                  BUNDLE/'libbeam_pynq.so', UART_DEVICE, I2C_DEVICE)
session.run(max_frames=10)

## 查看实际采样耗时与首错
统计只包含最近最多 256 个实际采样，采集、C 检测/组帧、显示退休分别记录；不把网页动画刷新率当相机速度。无运行时不生成实测结论。

In [ ]:
session.report()